# BMCS2203 Artificial Intelligence
## River Water Quality Prediction: Safe or Unsafe
#### Group Members: Chang Han Yean (SVM), Elwin Goh Yao Zu (Random Forest), Kaizen Soh (Decision Tree)

This notebook prepares the `waterQuality.csv` dataset for supervised machine learning. The target variable is `is_safe`, where `1` means safe water and `0` means unsafe water.


# 1.0 Project Setup

This section prepares the libraries, constants, file paths, and random seed used throughout preprocessing.


## 1.1 Import Required Libraries

These libraries are used for data handling, dataset splitting, and feature selection.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import mutual_info_classif

print("Libraries imported successfully.")


Libraries imported successfully.


## 1.2 Define File Paths And Random State

A fixed random state makes sampling, splitting, and feature selection reproducible.


In [2]:
RANDOM_STATE = 42
DATA_PATH = Path("waterQuality.csv")
OUTPUT_DIR = Path("DataTraining")

OUTPUT_DIR.mkdir(exist_ok=True)
np.random.seed(RANDOM_STATE)

print(f"Dataset path: {DATA_PATH}")
print(f"Output folder: {OUTPUT_DIR}")
print(f"Random state: {RANDOM_STATE}")


Dataset path: waterQuality.csv
Output folder: DataTraining
Random state: 42


# 2.0 Dataset Loading And Understanding

This section loads the dataset and checks its basic structure before preprocessing.


## 2.1 Load The Dataset

The dataset contains water contaminant measurements and the target column `is_safe`.


In [3]:
df = pd.read_csv(DATA_PATH)

print(f"Dataset shape: {df.shape}")


Dataset shape: (7999, 21)


## 2.2 Preview The Dataset

Displaying the first few rows helps confirm that the file has loaded correctly.


In [4]:
display(df.head())


,aluminium,ammonia,arsenic,barium,cadmium,chloramine,chromium,copper,flouride,bacteria,...,lead,nitrates,nitrites,mercury,perchlorate,radium,selenium,silver,uranium,is_safe
0,1.65,9.08,0.04,2.85,0.007,0.35,0.83,0.17,0.05,0.20,...,0.054,16.08,1.13,0.007,37.75,6.78,0.08,0.34,0.02,1
1,2.32,21.16,0.01,3.31,0.002,5.28,0.68,0.66,0.90,0.65,...,0.100,2.01,1.93,0.003,32.26,3.21,0.08,0.27,0.05,1
2,1.01,14.02,0.04,0.58,0.008,4.24,0.53,0.02,0.99,0.05,...,0.078,14.16,1.11,0.006,50.28,7.07,0.07,0.44,0.01,0
3,1.36,11.33,0.04,2.96,0.001,7.23,0.03,1.66,1.08,0.71,...,0.016,1.41,1.29,0.004,9.12,1.72,0.02,0.45,0.05,1
4,0.92,24.33,0.03,0.20,0.006,2.67,0.69,0.57,0.61,0.13,...,0.117,6.74,1.11,0.003,16.90,2.41,0.02,0.06,0.02,1


## 2.3 Check Columns And Data Types

This confirms the available features and identifies columns that may need numeric conversion.


In [5]:
print("Columns:")
print(df.columns.tolist())

print("\\nDataset information:")
display(df.info())


Columns:
['aluminium', 'ammonia', 'arsenic', 'barium', 'cadmium', 'chloramine', 'chromium', 'copper', 'flouride', 'bacteria', 'viruses', 'lead', 'nitrates', 'nitrites', 'mercury', 'perchlorate', 'radium', 'selenium', 'silver', 'uranium', 'is_safe']
\nDataset information:
<class 'pandas.DataFrame'>
RangeIndex: 7999 entries, 0 to 7998
Data columns (total 21 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   aluminium    7999 non-null   float64
 1   ammonia      7999 non-null   str    
 2   arsenic      7999 non-null   float64
 3   barium       7999 non-null   float64
 4   cadmium      7999 non-null   float64
 5   chloramine   7999 non-null   float64
 6   chromium     7999 non-null   float64
 7   copper       7999 non-null   float64
 8   flouride     7999 non-null   float64
 9   bacteria     7999 non-null   float64
 10  viruses      7999 non-null   float64
 11  lead         7999 non-null   float64
 12  nitrates     7999 non-null   float64
 1

None

## 2.4 Check Missing Values And Duplicates

Missing values and duplicate rows should be identified before model training.


In [6]:
print("Missing values per column:")
display(df.isna().sum())

print(f"Duplicate rows: {df.duplicated().sum()}")


Missing values per column:


aluminium      0
ammonia        0
arsenic        0
barium         0
cadmium        0
chloramine     0
chromium       0
copper         0
flouride       0
bacteria       0
viruses        0
lead           0
nitrates       0
nitrites       0
mercury        0
perchlorate    0
radium         0
selenium       0
silver         0
uranium        0
is_safe        0
dtype: int64

Duplicate rows: 0


## 2.5 Check Original Target Distribution

The target distribution shows whether the dataset is balanced or imbalanced.


In [7]:
print("Original target distribution:")
display(df["is_safe"].value_counts(dropna=False))


Original target distribution:


is_safe
0        7084
1         912
#NUM!       3
Name: count, dtype: int64

# 3.0 Data Cleaning

This section removes invalid target values, converts columns to numeric format, removes duplicates, and handles missing feature values.


## 3.1 Create A Working Copy

A copy is used so the original dataframe remains unchanged.


In [8]:
df_clean = df.copy()
print(f"Working copy shape: {df_clean.shape}")


Working copy shape: (7999, 21)


## 3.2 Convert Columns To Numeric Format

Invalid values such as `#NUM!` are converted to missing values (`NaN`) so they can be handled properly.


In [9]:
for col in df_clean.columns:
    df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce")

print("Columns converted to numeric format.")


Columns converted to numeric format.


### 3.2.1 Check Invalid Values After Conversion

After numeric conversion, invalid values appear as missing values.


In [10]:
print("Missing values after numeric conversion:")
display(df_clean.isna().sum())


Missing values after numeric conversion:


aluminium      0
ammonia        3
arsenic        0
barium         0
cadmium        0
chloramine     0
chromium       0
copper         0
flouride       0
bacteria       0
viruses        0
lead           0
nitrates       0
nitrites       0
mercury        0
perchlorate    0
radium         0
selenium       0
silver         0
uranium        0
is_safe        3
dtype: int64

## 3.3 Remove Invalid Target Rows

Rows with invalid or missing `is_safe` values cannot be used for supervised learning.


In [11]:
before_rows = len(df_clean)

df_clean = df_clean.dropna(subset=["is_safe"])
df_clean["is_safe"] = df_clean["is_safe"].astype(int)
df_clean = df_clean[df_clean["is_safe"].isin([0, 1])]

removed_rows = before_rows - len(df_clean)
print(f"Rows removed due to invalid target: {removed_rows}")
print(f"Dataset shape after target cleaning: {df_clean.shape}")


Rows removed due to invalid target: 3
Dataset shape after target cleaning: (7996, 21)


## 3.4 Remove Duplicate Rows

Duplicate records are removed to reduce repeated data from influencing the model.


In [12]:
before_dedup = len(df_clean)
df_clean = df_clean.drop_duplicates()

duplicates_removed = before_dedup - len(df_clean)
print(f"Duplicate rows removed: {duplicates_removed}")
print(f"Dataset shape after duplicate removal: {df_clean.shape}")


Duplicate rows removed: 0
Dataset shape after duplicate removal: (7996, 21)


## 3.5 Handle Missing Feature Values

Any remaining missing feature values are filled using the median value of each feature column.


In [13]:
feature_cols = [col for col in df_clean.columns if col != "is_safe"]

df_clean[feature_cols] = df_clean[feature_cols].fillna(df_clean[feature_cols].median())

print("Missing feature values handled using median imputation.")


Missing feature values handled using median imputation.


## 3.6 Verify Cleaned Dataset

This final check confirms that the cleaned dataset is ready for balancing.


In [14]:
print(f"Cleaned dataset shape: {df_clean.shape}")

print("\\nRemaining missing values:")
display(df_clean.isna().sum())

print("\\nCleaned target distribution:")
display(df_clean["is_safe"].value_counts())


Cleaned dataset shape: (7996, 21)
\nRemaining missing values:


aluminium      0
ammonia        0
arsenic        0
barium         0
cadmium        0
chloramine     0
chromium       0
copper         0
flouride       0
bacteria       0
viruses        0
lead           0
nitrates       0
nitrites       0
mercury        0
perchlorate    0
radium         0
selenium       0
silver         0
uranium        0
is_safe        0
dtype: int64

\nCleaned target distribution:


is_safe
0    7084
1     912
Name: count, dtype: int64

# 4.0 Data Balancing

The original dataset is imbalanced, so this section creates a balanced dataset with equal safe and unsafe records.


## 4.1 Separate Safe And Unsafe Records

Class `1` represents safe water and class `0` represents unsafe water.


In [15]:
safe_df = df_clean[df_clean["is_safe"] == 1]
unsafe_df = df_clean[df_clean["is_safe"] == 0]

print(f"Safe records: {len(safe_df)}")
print(f"Unsafe records: {len(unsafe_df)}")


Safe records: 912
Unsafe records: 7084


## 4.2 Sample 900 Records From Each Class

This creates a balanced 1,800-record dataset for the project requirement.


In [16]:
safe_sample = safe_df.sample(n=900, random_state=RANDOM_STATE)
unsafe_sample = unsafe_df.sample(n=900, random_state=RANDOM_STATE)

df_balanced = (
    pd.concat([unsafe_sample, safe_sample], axis=0)
    .sample(frac=1, random_state=RANDOM_STATE)
    .reset_index(drop=True)
)

print(f"Balanced dataset shape: {df_balanced.shape}")


Balanced dataset shape: (1800, 21)


## 4.3 Verify Balanced Dataset

The balanced dataset should contain 900 safe and 900 unsafe samples.


In [17]:
display(df_balanced["is_safe"].value_counts())


is_safe
1    900
0    900
Name: count, dtype: int64

# 5.0 Train/Test Split Before Feature Selection

The balanced data is split before feature selection to avoid data leakage. The test set must not influence which features are selected.


## 5.1 Prepare Full Feature Matrix And Target

At this point, all cleaned feature columns are kept. Feature selection will happen after the split using training data only.


In [18]:
X = df_balanced.drop(columns=["is_safe"])
y = df_balanced["is_safe"]

print(f"Full feature matrix shape: {X.shape}")
print(f"Target vector shape: {y.shape}")


Full feature matrix shape: (1800, 20)
Target vector shape: (1800,)


## 5.2 Split Into Training And Testing Sets

Stratified splitting keeps the class distribution equal in both training and testing sets.


In [19]:
X_train_full, X_test_full, y_train, y_test = train_test_split(
    X,
    y,
    train_size=1150,
    test_size=650,
    stratify=y,
    random_state=RANDOM_STATE,
)

print(f"X_train_full shape: {X_train_full.shape}")
print(f"X_test_full shape: {X_test_full.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")


X_train_full shape: (1150, 20)
X_test_full shape: (650, 20)
y_train shape: (1150,)
y_test shape: (650,)


## 5.3 Verify Train/Test Class Distribution

Both sets should remain balanced after splitting.


In [20]:
print("Training target distribution:")
display(y_train.value_counts())

print("Testing target distribution:")
display(y_test.value_counts())


Training target distribution:


is_safe
0    575
1    575
Name: count, dtype: int64

Testing target distribution:


is_safe
0    325
1    325
Name: count, dtype: int64

# 6.0 Feature Selection Using Training Data Only

Feature selection is part of the training pipeline. Therefore, Mutual Information is calculated using only the 1,150 training records, not the 650 test records.


## 6.1 Calculate Mutual Information Scores

Mutual Information measures how much each feature contributes to predicting `is_safe`. It can capture non-linear relationships.


In [21]:
mi_scores = mutual_info_classif(X_train_full, y_train, random_state=RANDOM_STATE)

feature_importance = (
    pd.DataFrame({"Feature": X_train_full.columns, "Mutual Information": mi_scores})
    .sort_values("Mutual Information", ascending=False)
    .reset_index(drop=True)
)

feature_importance.insert(0, "Rank", feature_importance.index + 1)
display(feature_importance)


,Rank,Feature,Mutual Information
0,1,aluminium,0.158182
1,2,cadmium,0.136395
2,3,chromium,0.073635
3,4,arsenic,0.070733
4,5,chloramine,0.069971
5,6,perchlorate,0.069967
6,7,radium,0.037779
7,8,nitrates,0.026949
8,9,barium,0.024956
9,10,bacteria,0.024167


### 6.1.1 Save Feature Importance Ranking

The ranking is saved for the report and for checking which features were selected.


In [22]:
feature_importance.to_csv(OUTPUT_DIR / "feature_mutual_information.csv", index=False)

print("Saved feature ranking to:")
print(OUTPUT_DIR / "feature_mutual_information.csv")


Saved feature ranking to:
DataTraining\feature_mutual_information.csv


## 6.2 Remove Features With Zero Mutual Information

Features with Mutual Information greater than 0 are selected. Features with Mutual Information equal to 0 are removed because they show no measured relationship with is_safe in the training data.


In [23]:
selected_features = feature_importance.loc[
    feature_importance["Mutual Information"] > 0, "Feature"
].tolist()

removed_features = feature_importance.loc[
    feature_importance["Mutual Information"] == 0, "Feature"
].tolist()

print(f"Selected {len(selected_features)} features (Mutual Information > 0):")
for feature in selected_features:
    print(f"- {feature}")

print(f"\\nRemoved {len(removed_features)} features (Mutual Information = 0):")
for feature in removed_features:
    print(f"- {feature}")


Selected 15 features (Mutual Information > 0):
- aluminium
- cadmium
- chromium
- arsenic
- chloramine
- perchlorate
- radium
- nitrates
- barium
- bacteria
- silver
- selenium
- ammonia
- viruses
- uranium
\nRemoved 5 features (Mutual Information = 0):
- copper
- lead
- flouride
- mercury
- nitrites


## 6.3 Apply Selected Features To Train And Test Sets

The selected feature list is applied to both training and testing sets. The test set follows the feature list but does not decide it.


In [24]:
X_train_selected = X_train_full[selected_features]
X_test_selected = X_test_full[selected_features]

print(f"Selected training feature shape: {X_train_selected.shape}")
print(f"Selected testing feature shape: {X_test_selected.shape}")


Selected training feature shape: (1150, 15)
Selected testing feature shape: (650, 15)


# 7.0 Save Preprocessed Outputs

The saved files can be used directly in the model training notebook or section.


## 7.1 Combine Selected Features And Target

The target column is added back so the train and test datasets can be saved as complete CSV files.


In [25]:
train_df = X_train_selected.copy()
train_df["is_safe"] = y_train.values

test_df = X_test_selected.copy()
test_df["is_safe"] = y_test.values

print(f"Final training set shape: {train_df.shape}")
print(f"Final testing set shape: {test_df.shape}")


Final training set shape: (1150, 16)
Final testing set shape: (650, 16)


## 7.2 Save Cleaned, Balanced, Train, And Test Files

These outputs separate each important preprocessing stage. The final train/test files contain only selected features plus `is_safe`.


In [26]:
df_clean.to_csv(OUTPUT_DIR / "water_quality_cleaned.csv", index=False)
df_balanced.to_csv(OUTPUT_DIR / "water_quality_balanced_1800.csv", index=False)
train_df.to_csv(OUTPUT_DIR / "water_quality_train_1150.csv", index=False)
test_df.to_csv(OUTPUT_DIR / "water_quality_test_650.csv", index=False)

print("Saved preprocessed dataset files.")


Saved preprocessed dataset files.
